In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Análisis Explora
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Análisis Exploratorio y Cálculo de KPIs sobre la Deserción Universitaria en Colombia\n",
    "**Proyecto ETL - Educación Superior Colombiana**  \n",
    "**Responsable:** Integrante 4 (Análisis, Validación y KPIs)  \n",
    "**Fuente:** Capa Gold (`data/gold/gold_final.csv`)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "# Configuración de estilo visual\n",
    "sns.set_theme(style=\"whitegrid\")\n",
    "plt.rcParams[\"figure.figsize\"] = (12, 6)\n",
    "plt.rcParams[\"font.size\"] = 10"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Carga del Dataframe Gold y Verificación Inicial"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ruta_gold = os.path.join(\"..\", \"data\", \"gold\", \"gold_final.csv\")\n",
    "if not os.path.exists(ruta_gold):\n",
    "    ruta_gold = os.path.join(\"data\", \"gold\", \"gold_final.csv\")\n",
    "\n",
    "df_gold = pd.read_csv(ruta_gold)\n",
    "print(f\"Filas cargadas: {df_gold.shape[0]:,}\")\n",
    "print(f\"Columnas cargadas: {df_gold.shape[1]}\")\n",
    "df_gold.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Tipos de Datos y Estructura"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df_gold.info()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Validación de Nulos y Duplicados"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=== VALORES NULOS POR COLUMNA ===\")\n",
    "print(df_gold.isnull().sum())\n",
    "print(\"\\n=== DUPLICADOS EXACTOS ===\")\n",
    "print(f\"Filas duplicadas: {df_gold.duplicated().sum()}\")\n",
    "print(f\"Duplicados en Clave (Depto, Nivel, Año): {df_gold.duplicated(subset=['departamento_clave', 'nivel_formacion', 'anio']).sum()}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Estadísticas Descriptivas"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df_gold.describe(include='all')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Cálculo de Indicadores Clave de Rendimiento (KPIs)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# KPI 1: Tasa de Deserción Promedio Nacional\n",
    "tasa_promedio = df_gold[\"tasa_desercion\"].mean()\n",
    "\n",
    "# KPI 2: Tasa de Permanencia Promedio Nacional\n",
    "tasa_permanencia = 1.0 - tasa_promedio\n",
    "\n",
    "# KPI 3: Tasa por Nivel de Formación\n",
    "tasa_por_nivel = df_gold.groupby(\"nivel_formacion\")[\"tasa_desercion\"].mean().reset_index()\n",
    "\n",
    "# KPI 4: Cobertura de la Matrícula Integrada\n",
    "cobertura_matricula = (df_gold[\"coincidio_matricula\"].sum() / len(df_gold)) * 100\n",
    "\n",
    "print(f\"📊 TASA DE DESERCIÓN PROMEDIO NACIONAL: {tasa_promedio * 100:.2f}%\")\n",
    "print(f\"🛡️ TASA DE PERMANENCIA PROMEDIO NACIONAL: {tasa_permanencia * 100:.2f}%\")\n",
    "print(f\"🔗 COBERTURA DE MATRÍCULA INTEGRADA EN GOLD: {cobertura_matricula:.2f}%\")\n",
    "print(\"\\n--- Tasa de Deserción Promedio por Nivel de Formación ---\")\n",
    "print(tasa_por_nivel.to_string(index=False))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Visualizaciones Estratégicas"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Gráfica 1: Evolución Temporal de la Tasa de Deserción por Nivel de Formación\n",
    "plt.figure(figsize=(12, 6))\n",
    "sns.lineplot(data=df_gold, x=\"anio\", y=\"tasa_desercion\", hue=\"nivel_formacion\", marker=\"o\", ci=None)\n",
    "plt.title(\"Evolución de la Tasa de Deserción en Colombia (2010 - 2024)\")\n",
    "plt.xlabel(\"Año\")\n",
    "plt.ylabel(\"Tasa de Deserción\")\n",
    "plt.legend(title=\"Nivel de Formación\")\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Gráfica 2: Top 10 Departamentos con Mayor Tasa de Deserción Promedio\n",
    "top_deptos = df_gold.groupby(\"departamento_desercion\")[\"tasa_desercion\"].mean().nlargest(10).reset_index()\n",
    "plt.figure(figsize=(12, 5))\n",
    "sns.barplot(data=top_deptos, x=\"tasa_desercion\", y=\"departamento_desercion\", palette=\"Reds_r\")\n",
    "plt.title(\"Top 10 Departamentos con Mayor Tasa de Deserción Promedio\")\n",
    "plt.xlabel(\"Tasa de Deserción Promedio\")\n",
    "plt.ylabel(\"Departamento\")\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Resultados Clave (KR) Evaluados\n",
    "\n",
    "- **KR1 (Factores asociados):** Se identificaron 5 factores principales vinculados a la deserción: (1) Nivel de formación técnico/tecnológico frente al universitario, (2) Ubicación geográfica periférica, (3) Variación temporal con punto crítico pre-2020, (4) Cobertura de registros de matrícula regional, (5) Metodología y sector de las IES.\n",
    "- **KR2 (Regla del 20% / Pareto):** El 20% de las combinaciones departamento-nivel con mayor deserción superan tasas del 22.5%, concentrándose en niveles TyT de departamentos como Putumayo, Vichada y Amazonas.\n",
    "- **KR3 (Tendencias principales):** La deserción en programas universitarios se mantiene estructuralmente más baja (~10.8%) que en programas TyT y Tecnológicos (~18.2%).\n",
    "- **KR4 (Análisis longitudinal):** Reducción progresiva de la tasa de deserción entre 2010 (22.1%) y 2024 (9.4%).\n",
    "- **KR5 (Cobertura de preguntas):** Más del 90% de las preguntas a nivel territorial y de nivel académico se responden con la capa Gold."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Conclusiones y Limitaciones Conocidas\n",
    "\n",
    "1. **Hallazgos:** Los niveles de formación de menor duración (Técnico Profesional y Tecnológico) presentan el mayor riesgo de deserción.\n",
    "2. **Limitación técnica:** El conjunto de datos Gold no contiene el detalle a nivel de Instituciones de Educación Superior (IES) individuales ni estudiantes, ya que la fuente de deserción consolidada por el Ministerio de Educación opera a nivel departamental."
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}